In [1]:
#import libraries
from pathlib import Path
import sqlite3
import json

import pandas as pd
import requests

#define global variables
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"

In [3]:
#create source dataset
participants = pd.DataFrame({
    "id": ["001", "002", "003", "004", "005"],
    "age": [24, 31, 27, 42, 35],
    "group": ["A", "B", "A", "B", "A"],
    "score": [81, 94, 88, 91, 85]
})

participants

,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


In [4]:
participants.dtypes

id       object
age       int64
group    object
score     int64
dtype: object

The ID and group are strings, while age and score are integers. Even if it's numeric, an ID cannot be averaged or have numerical summaries be performed on it, because it would be meaningless, so it should be a string that would be treated as an independent grouping instead.

In [14]:
csv_path = Path("../" + str(RAW_DIR) + "/participants.csv")

participants.to_csv(
    csv_path,
    index=False
)

csv_path.exists()

True

In the CSV, it saves the values only separated by commas. No types are saved, but each column does retain that type's qualities. 

In [16]:
#read in the df
loaded = pd.read_csv(csv_path)

loaded

,id,age,group,score
0,1,24,A,81
1,2,31,B,94
2,3,27,A,88
3,4,42,B,91
4,5,35,A,85


In [18]:
loaded.dtypes

id        int64
age       int64
group    object
score     int64
dtype: object

The ID is read in as an integer, so zero padding is removed for sake of simplicity. This is different from previously, where they were strings.

In [19]:
loaded = pd.read_csv(
    csv_path,
    dtype={
        "id": "string",
        "group": "category"
    }
)

loaded

,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


In [20]:
loaded.dtypes

id       string[python]
age               int64
group          category
score             int64
dtype: object

ID is now a string again, with the zero padding, and the group variable is now a cateogry, which can be subset according to unique values.

In [23]:
messy = pd.DataFrame({
    "id": ["001", "002", "003", "004"],
    "score": ["81", "-999", "88", "NA"]
})

messy_path = Path( "../" + str(RAW_DIR) + "/messy_scores.csv")

messy.to_csv(
    messy_path,
    index=False
)

#read the data back in
messy_loaded = pd.read_csv(
    messy_path,
    na_values=["NA", "-999"]
)

In [24]:
messy_loaded

,id,score
0,1,81.0
1,2,NaN
2,3,88.0
3,4,NaN


In [25]:
messy_loaded.dtypes

id         int64
score    float64
dtype: object

In [26]:
messy_loaded.isna().sum()

id       0
score    2
dtype: int64

Loading dataframes can misassign missing values, so it's necessary to inspect both the data file and the resulting read-in dataframe to make sure all data is being handled properly.

In [27]:
loaded.shape
loaded.head()
loaded.dtypes
loaded.isna().sum()
loaded["id"].nunique()
loaded["group"].value_counts()

group
A    3
B    2
Name: count, dtype: int64

In [28]:
assert len(loaded) == 5
assert loaded["id"].is_unique

In [29]:
assert set(
    loaded["group"].astype(str)
) <= {"A", "B"}

In [39]:
# SQLite Database
db_path = Path( "../" + str(RAW_DIR) + "/study.db")

connection = sqlite3.connect(db_path)

In [40]:
participants.to_sql(
    "participants",
    connection,
    if_exists="replace",
    index=False
)

5

In [32]:
sql_df = pd.read_sql(
    "SELECT * FROM participants",
    connection
)

sql_df

,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


Pandas can read in data, but the type is not stored, so it gives its best guess. This is different from an SQLite database, since the databse can store types and reading it in preserves that.

In [33]:
query = """
SELECT
    id,
    age,
    score
FROM participants
"""

selected = pd.read_sql(
    query,
    connection
)

In [34]:
participants[
    ["id", "age", "score"]
]

,id,age,score
0,001,24,81
1,002,31,94
2,003,27,88
3,004,42,91
4,005,35,85


In [36]:
query = """
SELECT *
FROM participants
WHERE age >= 30
"""

older = pd.read_sql(
    query,
    connection
)

participants.loc[
    participants["age"] >= 30
]

,id,age,group,score
1,002,31,B,94
3,004,42,B,91
4,005,35,A,85


The commands form the same outputs, but the Pandas commands are more straightforward. This of course depends on what data storgae is possible for a given scenario. SQL might be easier for a large databse, however, since it doesn't need to be stored locally compared to a dataframe.

In [41]:
query = """
SELECT
    group,
    COUNT(*) AS n,
    AVG(score) AS mean_score
FROM participants
GROUP BY group
"""

summary = pd.read_sql(
    query,
    connection
)

DatabaseError: Execution failed on sql '
SELECT
    group,
    COUNT(*) AS n,
    AVG(score) AS mean_score
FROM participants
GROUP BY group
': near "group": syntax error

In [ ]:
query = """
SELECT *
FROM participants
WHERE group = ?
"""

group_a = pd.read_sql(
    query,
    connection,
    params=("A",)
)

Parameters allow commands to be altered without needing to alter the command itself, which can be helpful for looping or large queries.

In [43]:
outcomes = pd.DataFrame({
    "id": ["001", "002", "003", "005"],
    "followup_score": [85, 97, 91, 89]
})

outcomes.to_sql(
    "outcomes",
    connection,
    if_exists="replace",
    index=False
)

4

In [44]:
query = """
SELECT
    p.id,
    p.age,
    p.group,
    p.score,
    o.followup_score
FROM participants AS p
LEFT JOIN outcomes AS o
    ON p.id = o.id
"""

joined = pd.read_sql(
    query,
    connection
)

DatabaseError: Execution failed on sql '
SELECT
    p.id,
    p.age,
    p.group,
    p.score,
    o.followup_score
FROM participants AS p
LEFT JOIN outcomes AS o
    ON p.id = o.id
': near "group": syntax error